# 📘 W7 Python Lab — 멀티 에이전트 오케스트레이션

**n8n W7 강의의 Python 버전.** Orchestrator + 3 Specialist 에이전트 팀 구조 구현.

## 학습 목표
- **Tool Use API**로 Claude가 스스로 도구 호출
- 서브 에이전트(News/OnChain/Macro)를 각각 함수로 구현
- Orchestrator가 질문 분석 → 필요한 서브 에이전트만 호출
- 메시지 버스 로깅 → 디버깅용 추적

## 🛠 사전 준비

```bash
pip install anthropic requests python-dotenv
```

기존 `.env`에 추가:
```
ALPHA_VANTAGE_KEY=여러분의_키
FRED_API_KEY=W1에서_쓰던_키
```

> 💡 **n8n AI Agent Tool 노드 = Python Tool Use API.** Claude가 어떤 함수를 언제 호출할지 스스로 결정합니다.

## 1. 환경 로드 + Claude 클라이언트

In [ ]:
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

claude = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
ALPHA_VANTAGE_KEY = os.getenv('ALPHA_VANTAGE_KEY')
FRED_API_KEY = os.getenv('FRED_API_KEY')

print('✓ Claude + API 키 준비 완료')

## 2. 3개 서브 에이전트 = 3개 Python 함수

각 에이전트는 **독립된 함수 + 고유 데이터 소스**. n8n의 AI Agent Tool 노드 역할.

In [ ]:
# === Agent 1: News Agent ===
def news_agent(ticker: str, hours_back: int = 24) -> dict:
    """뉴스 전문 에이전트 — Alpha Vantage NEWS_SENTIMENT."""
    url = 'https://www.alphavantage.co/query'
    params = {
        'function': 'NEWS_SENTIMENT',
        'tickers': ticker,
        'apikey': ALPHA_VANTAGE_KEY,
        'limit': 10
    }
    
    try:
        response = requests.get(url, params=params, timeout=10).json()
        feed = response.get('feed', [])
        
        if not feed:
            return {'error': 'no news found', 'ticker': ticker}
        
        scores = [float(a.get('overall_sentiment_score', 0)) for a in feed]
        avg_sentiment = sum(scores) / len(scores) if scores else 0
        
        return {
            'ticker': ticker,
            'sentiment_avg': round(avg_sentiment, 3),
            'article_count': len(feed),
            'top_headlines': [{
                'title': a['title'][:100],
                'score': round(float(a.get('overall_sentiment_score', 0)), 3)
            } for a in feed[:3]]
        }
    except Exception as e:
        return {'error': str(e), 'ticker': ticker}

# 테스트
print('News Agent 테스트:')
print(json.dumps(news_agent('AAPL'), indent=2, ensure_ascii=False))

In [ ]:
# === Agent 2: OnChain Agent ===
def onchain_agent(asset: str = 'bitcoin') -> dict:
    """온체인 전문 에이전트 — Blockchair."""
    asset_map = {'bitcoin': 'bitcoin', 'ethereum': 'ethereum', 'BTC': 'bitcoin', 'ETH': 'ethereum'}
    chain = asset_map.get(asset.lower(), asset.lower())
    
    url = f'https://api.blockchair.com/{chain}/stats'
    
    try:
        response = requests.get(url, timeout=10).json()
        d = response['data']
        
        return {
            'asset': chain,
            'transactions_24h': d.get('transactions_24h'),
            'market_price_usd': d.get('market_price_usd'),
            'mempool_transactions': d.get('mempool_transactions'),
            'dominant_signal': (
                'HIGH_ACTIVITY' if d.get('transactions_24h', 0) > 400000
                else 'NORMAL_ACTIVITY'
            )
        }
    except Exception as e:
        return {'error': str(e), 'asset': chain}

# 테스트
print('\nOnChain Agent 테스트:')
print(json.dumps(onchain_agent('bitcoin'), indent=2, ensure_ascii=False))

In [ ]:
# === Agent 3: Macro Agent ===
def macro_agent(focus: str = 'rates') -> dict:
    """거시경제 전문 에이전트 — FRED."""
    series_map = {
        'rates': 'DFF',          # 연방기금 금리
        'inflation': 'CPIAUCSL', # CPI
        'employment': 'UNRATE',  # 실업률
        'vix': 'VIXCLS'          # VIX
    }
    series = series_map.get(focus, 'DFF')
    
    url = 'https://api.stlouisfed.org/fred/series/observations'
    params = {
        'series_id': series,
        'api_key': FRED_API_KEY,
        'file_type': 'json',
        'sort_order': 'desc',
        'limit': 6
    }
    
    try:
        response = requests.get(url, params=params, timeout=10).json()
        obs = response['observations']
        
        latest = float(obs[0]['value'])
        six_ago = float(obs[-1]['value']) if len(obs) >= 6 else latest
        
        trend = 'UP' if latest > six_ago * 1.02 else ('DOWN' if latest < six_ago * 0.98 else 'FLAT')
        
        return {
            'focus_area': focus,
            'series': series,
            'current_value': latest,
            'trend_6m': trend,
            'latest_date': obs[0]['date']
        }
    except Exception as e:
        return {'error': str(e), 'focus': focus}

# 테스트
print('\nMacro Agent 테스트:')
print(json.dumps(macro_agent('rates'), indent=2, ensure_ascii=False))

## 3. Claude Tool Use — 핵심 개념

Orchestrator Claude에게 **도구 목록**(JSON 스키마)을 알려주면, Claude가 질문을 분석해서 어느 도구를 호출할지 결정합니다.

In [ ]:
# 각 서브 에이전트의 도구 스키마 정의
# Claude가 이 설명을 보고 언제 어떻게 호출할지 결정

TOOLS = [
    {
        'name': 'news_agent',
        'description': (
            '주식/암호화폐에 대한 최근 뉴스와 센티먼트를 조회합니다. '
            '"애플 뉴스", "삼성전자 최근 이슈" 같은 질문에 사용. '
            'AAPL, MSFT, NVDA 등 ticker를 입력.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'ticker': {'type': 'string', 'description': '종목 티커 (예: AAPL)'},
                'hours_back': {'type': 'integer', 'default': 24}
            },
            'required': ['ticker']
        }
    },
    {
        'name': 'onchain_agent',
        'description': (
            'Bitcoin 또는 Ethereum의 온체인 네트워크 활동 조회. '
            '"BTC 고래 움직임", "이더리움 트랜잭션 수" 같은 암호화폐 질문에만 사용.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'asset': {'type': 'string', 'enum': ['bitcoin', 'ethereum']}
            },
            'required': ['asset']
        }
    },
    {
        'name': 'macro_agent',
        'description': (
            '거시경제 지표 조회 (금리, 인플레이션, 실업률, VIX). '
            '"연준 금리", "인플레이션 추세", "변동성 지수" 같은 매크로 질문에 사용.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'focus': {'type': 'string', 'enum': ['rates', 'inflation', 'employment', 'vix']}
            },
            'required': ['focus']
        }
    }
]

TOOL_FUNCTIONS = {
    'news_agent': news_agent,
    'onchain_agent': onchain_agent,
    'macro_agent': macro_agent
}

print(f'✓ {len(TOOLS)}개 도구 스키마 준비 완료')

## 4. Orchestrator — Tool Use Agent Loop

Claude가 도구 호출 결정 → 실행 → 결과 피드백 → 최종 답변 루프.

In [ ]:
ORCHESTRATOR_SYSTEM = """당신은 리서치팀 팀장입니다. 3명의 전문 에이전트를 조율합니다.

팀 구성:
- news_agent: 뉴스·센티먼트 전문
- onchain_agent: 암호화폐 온체인 데이터 (BTC/ETH만)
- macro_agent: 거시경제 지표 (금리·인플레이션·VIX)

워크플로:
1. 사용자 질문 분석 → 관련 에이전트만 호출 (불필요한 호출 = 비용 낭비)
2. 암호화폐 질문: onchain + macro
3. 주식 질문: news + macro
4. 복잡한 질문: 3개 모두 호출 후 종합

답변 형식:
- 각 에이전트 결과를 출처 명시 ("News Agent에 따르면...")
- 최종 verdict: BUY / WATCH / HOLD / AVOID + confidence 1~5
- 2-3문장 종합 근거
- 교육 목적임을 명시"""

def orchestrator(user_query: str, verbose: bool = True) -> dict:
    """멀티 에이전트 오케스트레이션 메인 함수.
    
    Tool Use agent loop:
    1. Claude가 도구 호출 요청
    2. 도구 실제 실행
    3. 결과를 Claude에 피드백
    4. 추가 호출 필요하면 반복, 아니면 최종 답변
    """
    messages = [{'role': 'user', 'content': user_query}]
    message_bus = []
    iteration = 0
    
    while iteration < 5:  # 안전장치
        iteration += 1
        if verbose:
            print(f'\n--- Iteration {iteration} ---')
        
        response = claude.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=1500,
            system=ORCHESTRATOR_SYSTEM,
            tools=TOOLS,
            messages=messages
        )
        
        if response.stop_reason == 'end_turn':
            final_text = ''.join(
                b.text for b in response.content if b.type == 'text'
            )
            message_bus.append({
                'actor': 'orchestrator', 'action': 'FINAL',
                'content': final_text[:200]
            })
            return {
                'answer': final_text,
                'message_bus': message_bus,
                'iterations': iteration
            }
        
        if response.stop_reason != 'tool_use':
            break
        
        # 도구 호출 실행
        messages.append({'role': 'assistant', 'content': response.content})
        tool_results = []
        
        for block in response.content:
            if block.type != 'tool_use':
                continue
            
            tool_name = block.name
            tool_input = block.input
            
            if verbose:
                print(f'  → Calling {tool_name}({tool_input})')
            message_bus.append({
                'actor': 'orchestrator', 'action': 'CALL',
                'target': tool_name, 'input': tool_input
            })
            
            result = TOOL_FUNCTIONS[tool_name](**tool_input)
            
            if verbose:
                print(f'  ← Result: {json.dumps(result, ensure_ascii=False)[:100]}')
            message_bus.append({
                'actor': tool_name, 'action': 'RESPOND',
                'content': str(result)[:150]
            })
            
            tool_results.append({
                'type': 'tool_result',
                'tool_use_id': block.id,
                'content': json.dumps(result)
            })
        
        messages.append({'role': 'user', 'content': tool_results})
    
    return {
        'answer': 'Max iterations reached',
        'message_bus': message_bus,
        'iterations': iteration
    }

## 5. 테스트 — 3가지 유형의 질문

In [ ]:
# 질문 1: 주식 → news + macro 호출 예상
print('=' * 70)
print('🧪 테스트 1: 주식 (news + macro 호출 예상)')
print('=' * 70)
result1 = orchestrator('AAPL 요즘 어때? 매수해도 될까?')
print(f'\n📝 답변:\n{result1["answer"]}')

In [ ]:
# 질문 2: 암호화폐 → onchain + macro 호출 예상
print('\n' + '=' * 70)
print('🧪 테스트 2: 암호화폐 (onchain + macro 호출 예상)')
print('=' * 70)
result2 = orchestrator('비트코인 지금 사도 될까? 온체인 데이터 기반으로 분석해줘')
print(f'\n📝 답변:\n{result2["answer"]}')

In [ ]:
# 질문 3: 순수 매크로 → macro만 호출 예상
print('\n' + '=' * 70)
print('🧪 테스트 3: 매크로 (macro만 호출 예상)')
print('=' * 70)
result3 = orchestrator('미국 금리가 최근 어떻게 움직이고 있어?')
print(f'\n📝 답변:\n{result3["answer"]}')

## 6. 메시지 버스 시각화

에이전트들이 어떻게 대화했는지 타임라인으로 확인.

In [ ]:
def print_message_bus(bus: list[dict]):
    """메시지 버스를 읽기 쉬운 타임라인으로 출력."""
    icons = {
        'orchestrator': '🎼',
        'news_agent': '📰',
        'onchain_agent': '⛓️',
        'macro_agent': '🌍'
    }
    
    for i, msg in enumerate(bus):
        icon = icons.get(msg['actor'], '❓')
        action = msg['action']
        
        if action == 'CALL':
            print(f'{i+1:2}. {icon} Orchestrator → {msg["target"]}')
            print(f'      입력: {msg["input"]}')
        elif action == 'RESPOND':
            print(f'{i+1:2}. {icon} {msg["actor"]} 응답')
            print(f'      {msg["content"]}...')
        elif action == 'FINAL':
            print(f'{i+1:2}. {icon} 최종 답변')
            print(f'      {msg["content"]}...')
        print()

print('📊 테스트 1 (주식) 메시지 버스:\n')
print_message_bus(result1['message_bus'])
print(f'\n⏱ 총 iteration: {result1["iterations"]}')

## 🎯 과제

1. **4번째 에이전트 추가** — `korean_stock_agent`. pykrx 라이브러리로 한국 종목 전용 구현
2. **병렬 호출 실험** — `asyncio.gather()`로 여러 에이전트를 동시 실행 → 응답 시간 측정 (순차 대비 2~3배 빠름)
3. **메시지 버스 CSV 저장** — `pandas.DataFrame(message_bus).to_csv()`로 장기 로그 축적
4. **질문 10개 테스트 세트** — 다양한 질문으로 Orchestrator가 적절한 에이전트를 호출하는지 검증
5. **Tool Description 튜닝** — 잘못된 에이전트가 호출되면 description 수정해서 정확도 향상

## 🔄 n8n vs Python 비교

| 개념 | n8n | Python |
|---|---|---|
| 서브 에이전트 | AI Agent Tool 노드 | Python 함수 |
| Tool Description | 노드 Description 필드 | `description` in TOOLS |
| Input Schema | 노드 Input 설정 | `input_schema` JSON Schema |
| Orchestrator | 상위 AI Agent 노드 | `claude.messages.create(tools=...)` |
| Tool Use Loop | 내부 자동 | while 루프 수동 구현 |
| Intermediate Steps | `Return Intermediate Steps` 옵션 | `response.content`의 tool_use 블록 |

**핵심 배움 — Tool Use의 본질:**
1. Claude는 함수를 **직접 실행 못 함**. 단지 "이 함수를 이렇게 호출해주세요"라고 요청할 뿐
2. **실제 실행은 우리 코드**가 함 (while 루프)
3. 결과를 다시 Claude에 보내면 Claude가 다음 행동 결정
4. 이 왕복이 **Agent Loop**의 실체

n8n이 이 loop를 시각적으로 숨기는 것뿐 — 내부 원리는 Python에서 쓴 것과 동일합니다.